# Consequence direction — Phases 2 to 5 (Colab Pro)

Every expensive step lives in a script under `scripts/`, runs as a subprocess, and stores its
numbers in `artifacts/results/<step>.json`. This notebook **runs steps and renders their stored
results** — it does not compute anything itself.

Two consequences worth knowing before you start:

* **Re-running a cell is free.** A step whose inputs, parameters and code are unchanged prints
  `reusing …` and returns the saved numbers. Nothing recomputes by accident.
* **A disconnect costs almost nothing.** No result lives in kernel state, so a fresh runtime
  restores `artifacts/` from Drive and picks up wherever it left off. The only variable this
  notebook carries between cells is the repo path.

A step recomputes only when something it depends on actually moved — an edited dataset, a
re-selected layer, a fixed bug in the analysis — and it says which. That is the point: the
failure this guards against is not slowness, it is a number in the write-up that no longer
matches the inputs it claims to come from.

**Before running:** `Runtime → Change runtime type → GPU` (L4 is ideal on Pro), then run top to
bottom. GPU is needed for sections 2, 3, 5, 11, 16 and 17; every other step is CPU against cached
artifacts, so the whole Phase 3/4 analysis re-runs on a CPU runtime.

## 0 · Runtime, dependencies, repo

In [ ]:
import shutil, subprocess, torch
if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip())
print("CUDA:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else
      "NO GPU — fine for every CPU step; sections 2, 3, 5, 11, 16, 17 need one")

In [ ]:
!pip -q install "transformers>=4.44" accelerate jaxtyping einops datasets
print("deps ready")

**Getting the repo onto the runtime.** The Colab runtime is remote and ephemeral, so the repo
has to arrive somehow. This auto-detects it if it is already present (cloned, synced via Drive,
or mounted by the VS Code Colab extension) and clones it otherwise.

In [ ]:
import os, sys, glob, zipfile

GIT_URL      = "https://github.com/AbiOppenheim/consequence-awareness.git"
ALLOW_UPLOAD = False  # True ONLY in the Colab web UI — the picker widget hangs in VS Code
MOUNT_DRIVE  = False  # True to mount Drive (repo under /content/drive/MyDrive/...)

if MOUNT_DRIVE:
    from google.colab import drive; drive.mount("/content/drive")

def find_repo():
    p = os.path.abspath(os.getcwd())                     # 1) walk up from cwd
    while p != "/":
        if os.path.exists(os.path.join(p, "configs", "qwen.yaml")):
            return p
        p = os.path.dirname(p)
    for root in ("/content/drive/MyDrive", "/content"):  # 2) common runtime locations
        hits = glob.glob(os.path.join(root, "**", "configs", "qwen.yaml"), recursive=True)
        if hits:
            return os.path.dirname(os.path.dirname(hits[0]))
    return None

REPO = find_repo()
if REPO is None and GIT_URL:
    os.system(f"git clone --depth 1 {GIT_URL} /content/consequence-awareness")
    REPO = find_repo()
if REPO is None and ALLOW_UPLOAD:
    from google.colab import files
    up = files.upload()
    with zipfile.ZipFile(next(iter(up))) as z:
        z.extractall("/content")
    REPO = find_repo()
if REPO is None:
    raise SystemExit(
        "Repo not on this runtime. Choose one and re-run this cell:\n"
        "  - set GIT_URL to your repo (works in VS Code and the web UI), or\n"
        "  - set MOUNT_DRIVE=True with the repo in your Drive, or\n"
        "  - set ALLOW_UPLOAD=True *only in the Colab web UI* and upload the zip.\n"
        "files.upload() hangs forever in VS Code — its picker needs the Colab web frontend.")

os.chdir(REPO); sys.path.insert(0, os.path.join(REPO, "src"))
print("REPO =", REPO)

### The two helpers this notebook runs on

`step(...)` runs a pipeline step as a subprocess and **raises** on failure — a bare `!python`
prints its error and lets the cell report success, which is how a broken stage 02 once went
unnoticed while every later section quietly read directions nothing had minted.

`R(...)` loads a stored result. Plot cells read from it, so a figure can never disagree with
the numbers the step printed.

In [ ]:
import subprocess, sys
import matplotlib.pyplot as plt
from consequence import results

def step(script, *args, force=False):
    """Run scripts/<script>. Cached steps are a no-op; a failure stops the cell."""
    cmd = [sys.executable, f"scripts/{script}", *map(str, args)] + (["--force"] if force else [])
    p = subprocess.run(cmd, cwd=REPO, text=True, capture_output=True)
    print(p.stdout.rstrip())
    if p.returncode:
        print("STDERR:\n", p.stderr[-4000:])
        raise RuntimeError(f"{script} failed (exit {p.returncode}) — see STDERR above")

def R(name):
    """The stored result of a step, as a plain dict."""
    return results.load(name)[0]

print("helpers ready")

### Drive sync — do this early, not once the runtime is already dying

The activation caches (~800 MB) and every direction live only on this machine. Backing them up
turns a recycled runtime into a 30-second restore instead of another GPU session. **Restore
first**: if a previous session already computed a step, this brings back both the artifact and
its stored result, and every step below becomes a no-op.

In [ ]:
MOUNT = True     # set False to skip Drive entirely
DEST  = "/content/drive/MyDrive/consequence-awareness-artifacts"

import os, shutil
def sync(src_root, dst_root):
    n = 0
    for dirpath, _, files in os.walk(src_root):
        for f in files:
            s = os.path.join(dirpath, f)
            d = os.path.join(dst_root, os.path.relpath(s, src_root))
            os.makedirs(os.path.dirname(d), exist_ok=True)
            if not os.path.exists(d) or os.path.getmtime(s) > os.path.getmtime(d):
                shutil.copy2(s, d); n += 1
    return n

if MOUNT:
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    if os.path.exists(DEST):
        print("restored", sync(DEST, "artifacts"), "files from Drive")
    else:
        print("no Drive backup yet — will be created by the backup cell below")

---
# Phase 2 — the r̂ gate

Validate the extraction machinery by reproducing **Arditi's refusal direction** on Qwen2.5 and
checking that the direction *our* code extracts points the same way. Two random directions in
3,584-D sit at |cos| ≈ 0.017, so ≈ 1 is unambiguous.

The gate is near-tautological by design — both sides compute the same math on the same prompts
— so it catches gross plumbing errors (caching, token position, layer indexing, chat template)
and nothing else. A methods footnote, not a finding. But if it fails, nothing downstream is
worth computing.

> On the pod we run Arditi in its own venv (Rule 1). On Colab we keep the boundary by running
> each heavy step as a **separate subprocess**: our code never imports theirs in-process, and
> each model load frees when its process exits.

## 1 · Clone Arditi and build the shared prompt file

In [ ]:
import os, subprocess
ARDITI = os.path.join(REPO, "external", "refusal_direction")
if not os.path.exists(ARDITI):
    subprocess.run(["git","clone","--depth","1",
                    "https://github.com/andyrdt/refusal_direction", ARDITI], check=True)
print("splits:", os.listdir(os.path.join(ARDITI, "dataset", "splits")))
step("phase2_build_refusal.py")
!head -1 data/contrast/refusal.jsonl

## 2 · Cache activations with OUR code — GPU
**First run downloads Qwen2.5-7B (~15 GB).** Skips itself if the `.npz` is already present and
was built from the same dataset file.

In [ ]:
step("01_cache_acts.py", "--dataset", "refusal")
!ls -la artifacts/activations/

## 3 · Run Arditi's extraction on the SAME prompts — GPU
Their unchanged `generate_directions`, fed our prompts through the Qwen2.5 adapter. Reuses the
model already in the HF cache. Writes the diff-in-means cube `[n_positions, n_layers, d_model]`.

In [ ]:
import os, subprocess, sys
if os.path.exists("artifacts/directions/r_hat_mean_diffs.pt"):
    print("[skip] r_hat_mean_diffs.pt already present")
else:
    p = subprocess.run(
        [sys.executable, "../../scripts/arditi_qwen25/run_extract.py",
         "--refusal", "../../data/contrast/refusal.jsonl",
         "--model",   "Qwen/Qwen2.5-7B-Instruct",
         "--out",     "../../artifacts/directions/r_hat_mean_diffs"],
        cwd=os.path.join(REPO, "external", "refusal_direction"),
        env=dict(os.environ, PYTHONPATH="."), text=True, capture_output=True)
    print(p.stdout[-4000:])
    if p.returncode != 0:
        print("STDERR:\n", p.stderr[-4000:])
        raise SystemExit(f"Arditi extraction failed (exit {p.returncode})")
    print("OK -> artifacts/directions/r_hat_mean_diffs.pt")

## 4 · The gate + its red-team — CPU

`02b_gate.py` computes the offset sweep **and** the four checks that make a high cosine mean
something, in one step, because a red-team in a separate cell is a red-team someone skips:

1. **cross-layer** — does offset +1 actually beat the others, or is everything high?
2. **self-similarity** — are our own adjacent layers already ≈1? then the test has no power.
3. **permutation** — shuffle the harmful/harmless labels; a sound pipeline must collapse to
   noise. This is the decisive one.
4. **outlier dominance** — is the direction just a few huge residual dimensions?

It also mints `r_hat_L*.pt` with Rule-2 sidecars, so the geometry step and the sweep's `r_hat`
reference condition have directions to load.

In [ ]:
step("02b_gate.py")

In [ ]:
g = R("gate")
c = [r["cos"] for r in g["per_layer_offset1"]]
x = [r["our_layer"] for r in g["per_layer_offset1"]]
plt.figure(figsize=(8,4))
plt.axhspan(-g["random_direction_abs_cos_p95"], g["random_direction_abs_cos_p95"],
            color="gray", alpha=.2, label="random-direction band")
plt.plot(x, c, "o-", label="cos(our r̂, Arditi r̂) at offset +1")
plt.axhline(1, ls="--", color="green"); plt.ylim(-.1, 1.05)
plt.xlabel("our cache layer index"); plt.ylabel("cosine")
plt.title("r̂ gate — our extraction vs Arditi's"); plt.legend(); plt.grid(alpha=.3); plt.show()

---
# Phase 3 — does a consequence direction `v_C` exist?

The gate passed, so the machinery is trustworthy. Now the research question: is "this situation
is real vs. hypothetical" linearly represented, and does it **generalize to framing templates
the probe has never seen**?

**Discipline (CLAUDE.md §2).** Held-out is by *framing template*, never by example. The layer is
chosen by cross-validation **inside the training templates only**. Picking the layer that
maximises held-out accuracy is selecting on the test set and inflates the number you then report.

## 5 · Cache the consequence set — GPU

In [ ]:
step("01_cache_acts.py", "--dataset", "consequence")
!ls -la artifacts/activations/

## 6 · Extract `v_C` and the random null — CPU

`--split train` is the default and is not optional. Difference-in-means over *all* rows would
build `v_C` partly out of the held-out templates, and section 8 would then score that same
`v_C` on those rows and call the number generalization. Step 09 refuses to run against a `v_C`
whose sidecar does not say `split: train`.

In [ ]:
step("02_extract_directions.py", "--dataset", "consequence")

## 7 · Pick the layer using TRAIN templates only — CPU

Group-wise 5-fold CV where the groups are framing templates, so every fold validates on
templates the probe did not train on — a miniature of the real held-out test. The chosen layer
is written to `artifacts/results/layer_select.json`, and every later step reads it from there:
the held-out reveal, the red-team, the alpha calibration, and the sweep. Nothing downstream
depends on a variable being in scope in this kernel.

This is the most expensive CPU step (45 logistic regressions on 1428 × 3584 features), which is
exactly why it is stored.

In [ ]:
step("03_probe.py", "--dataset", "consequence")

In [ ]:
s = R("layer_select")
xs = [r["layer"] for r in s["by_layer"]]; ys = [r["train_cv_auc"] for r in s["by_layer"]]
plt.figure(figsize=(7,3.5))
plt.plot(xs, ys, "o-"); plt.axvline(s["best_layer"], ls="--", color="green",
                                    label=f"selected L{s['best_layer']}")
plt.ylim(0.5, 1.02); plt.xlabel("layer"); plt.ylabel("train-template group-CV AUC")
plt.title(f"Layer selection (spread {s['spread']:.3f})"); plt.legend(); plt.grid(alpha=.3); plt.show()

## 8 · HELD-OUT REVEAL — run once

Everything above used training templates only. This evaluates the **frozen held-out framings**
at the layer already chosen.

Three readouts and the two baselines that make them mean anything: the **trained probe** (is the
information linearly readable on unseen framings?), the **`v_C` projection** (does the raw
direction transfer, with no fitting? — the stronger claim), the **random direction** (the null),
and the **BoW baseline** from `audit_contrast.py` (the honest surface-vocabulary number the
probe must beat).

If this result already exists and anything it depends on has changed, the step archives the
previous version as `heldout.prev-<sha>.json` and says so. Both numbers then go in the write-up,
with the second labelled post-hoc. Do not iterate against this number.

In [ ]:
step("09_heldout.py")

In [ ]:
h = R("heldout")
bars = [("trained probe", h["probe_auc"]), ("v_C projection", h["v_c_projection_auc"]),
        ("BoW baseline", h["bow_baseline_auc"]), ("random dir", h["random_direction_auc"])]
plt.figure(figsize=(7,3.5))
plt.bar([b[0] for b in bars], [b[1] for b in bars],
        color=["#3b7dd8","#3b7dd8","#999","#ccc"])
plt.axhline(0.5, ls="--", color="k", lw=1, label="chance")
plt.ylim(0, 1.05); plt.ylabel("held-out AUC")
plt.title(f"Held-out framings, L{h['layer']} (n={h['n_heldout_rows']}, "
          f"{h['n_heldout_templates']} unseen templates)")
plt.legend(); plt.grid(axis="y", alpha=.3); plt.show()

## 9 · Red-team the held-out result — CPU

Four checks, ordered by how likely each is to change the conclusion: per-route breakdown
(is the average hiding heterogeneity?), held-out at every layer (does the result hang on the
layer choice?), stronger text baselines (TF-IDF uni/bigram and char n-grams — a much fairer
proxy for "surface form" than bag-of-words), and the random null as a **distribution** rather
than a single draw.

In [ ]:
step("10_redteam_heldout.py")

In [ ]:
rt = R("heldout_redteam")
fig, ax = plt.subplots(1, 2, figsize=(13, 4))

rows = sorted(rt["per_route"], key=lambda r: r["auc"])
ax[0].barh([r["route"] for r in rows], [r["auc"] for r in rows],
           color=["#d9534f" if r["auc"] < 0.75 else "#3b7dd8" for r in rows])
ax[0].axvline(0.5, ls="--", color="k", lw=1); ax[0].set_xlim(0, 1.02)
ax[0].set_xlabel("held-out AUC (unfitted v_C)"); ax[0].set_title("Per held-out route")
ax[0].tick_params(labelsize=8)

xs = [r["layer"] for r in rt["by_layer"]]
ax[1].plot(xs, [r["probe_auc"] for r in rt["by_layer"]], "o-", label="trained probe")
ax[1].plot(xs, [r["v_c_projection_auc"] for r in rt["by_layer"]], "s-", label="unfitted v_C")
for b in rt["text_baselines"]:
    ax[1].axhline(b["auc"], ls=":", lw=1, color="gray")
ax[1].axhspan(rt["random_null"]["p05"], rt["random_null"]["p95"], color="gray", alpha=.2,
              label="random null 5-95%")
ax[1].axvline(rt["layer"], ls="--", color="green", lw=1)
ax[1].set_ylim(0.3, 1.02); ax[1].set_xlabel("layer"); ax[1].set_ylabel("held-out AUC")
ax[1].set_title("Held-out at every layer (dotted = text baselines)")
ax[1].legend(fontsize=8); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

## 10 · Back the artifacts up to Drive

In [ ]:
if MOUNT:
    print("backed up", sync("artifacts", DEST), "files ->", DEST)

---
# Phase 4 — `v_MP`: is `v_C` just the persona direction?

The sharpest alternative explanation for Phase 3: the model is not tracking "is this real", it
is adopting a **play-acting persona**, and `v_C` is that persona axis.

**Zhong's public repo cannot produce `v_MP` as shipped** — `extract_vectors.py` imports an
`ithou` package that is not in the release, and there is no `data/`. But their code pins the
exact Qwen2.5 configuration (`src/compliant_residual.py`):

    trait = compliant_v2 | vector = model_persona | position = prompt_end | layer = 20

`prompt_end` is our own token convention, so we rebuild that direction with **our** validated
difference-in-means, through the same `02_extract_directions.py` that made `v_C` — three
directions extracted three different ways could not be compared by cosine. Label it honestly:
*our reimplementation of Zhong's method using their released trait definitions*, not their
vector. Unlike `r_hat` there is no reference to check it against — a real limitation.

Two framings, because the choice is itself a confound:
* **`persona`** — instruction in the SYSTEM prompt (faithful to their `model_persona`)
* **`persona_ut`** — instruction in the USER turn (structurally identical to how `v_C` and
  `r_hat` were extracted, so a low cosine cannot be blamed on prompt structure)

## 11 · Build and cache both persona framings — GPU (~3 min)

In [ ]:
step("phase4_build_persona.py")
step("01_cache_acts.py", "--dataset", "persona")
step("01_cache_acts.py", "--dataset", "persona_ut")

## 12 · Extract `v_MP` — CPU

In [ ]:
step("02_extract_directions.py", "--dataset", "persona",    "--kind", "v_mp")
step("02_extract_directions.py", "--dataset", "persona_ut", "--kind", "v_mp")

## 13 · Geometry: is `v_C` its own axis? — CPU

Claim 1's distinctness half, over every swept layer rather than only the selected one — a
cosine that is small at one layer and large at another is a fact about the geometry that a
single number hides.

Sign convention: `v_MP` = mean(compliant) − mean(restrictive) points TOWARD compliance, roughly
opposite in spirit to `r_hat` = mean(harmful) − mean(harmless). Compare magnitudes; read the
sign only as which way the axis points.

In [ ]:
step("04_geometry.py")

In [ ]:
gm = R("geometry")
xs = gm["layers"]
plt.figure(figsize=(8,4))
for label in ["cos(v_C, r_hat)", "cos(v_C, v_MP sys)", "cos(v_C, v_MP ut)",
              "cos(r_hat, v_MP sys)", "cos(v_MP sys, v_MP ut)"]:
    ys = [r.get(label) for r in gm["by_layer"]]
    if any(y is not None for y in ys):
        plt.plot(xs, ys, "o-", label=label)
band = [r["null_abs_cos_p95"] for r in gm["by_layer"]]
plt.fill_between(xs, [-b for b in band], band, color="gray", alpha=.25,
                 label="random-direction band")
plt.axvline(gm["selected_layer"], ls="--", color="green", lw=1)
plt.axhline(0, color="k", lw=.8); plt.ylim(-1.05, 1.05)
plt.xlabel("layer"); plt.ylabel("cosine"); plt.title("Direction geometry across layers")
plt.legend(fontsize=8); plt.grid(alpha=.3); plt.show()

---
# Phase 5 — the causal test

Everything so far is **correlational**: `v_C` is readable, and distinct from `r_hat` and
persona. None of it shows the model *uses* it. This phase intervenes.

The design, fixed before running so nothing here is chosen after seeing results:

| condition | why it is in the sweep |
|---|---|
| baseline | attack success with no intervention |
| `+α·v_C` | the hypothesis: steering toward "real" restores refusal |
| `−α·v_C` | a real direction pushes **both** ways. If only `+α` moves anything, the likely story is "a large vector degrades output" |
| `+α·v_random` | the null. Non-negotiable |
| `+α·r_hat` | reference: how much can *any* direction do here? |

Scoring is **three-way** (refusal / bypass / degenerate), never a refusal rate alone: high
steering breaks fluency before it restores refusal, and a two-way metric scores gibberish as a
successful defense.

## 14 · Build the evaluation sets
Published attacks only — nothing here authors a jailbreak. XSTest (safe prompts) is the guard on
the causal claim; the fiction/role-play attacks come from a published corpus and are gitignored,
so they are downloaded rather than shipped in the repo.

In [ ]:
step("phase5_build_eval.py", "--all", "--limit", "100")
!wc -l data/eval/*.jsonl

## 15 · Calibrate α against measured residual norms — CPU

An α that is tiny relative to the residual stream does nothing; one that is huge only produces
gibberish. Expressing α as a fraction of the median activation norm **at the steering layer**
makes the ladder meaningful and reportable. The largest rung is EXPECTED to break fluency — the
coherence collapse point is itself a reported number.

The sweep reads this ladder with `--alphas auto`, so there is nothing to retype and no way to
sweep a ladder calibrated against a different layer.

In [ ]:
step("11_calibrate_alpha.py")

## 16 · Smoke test FIRST — 3 prompts, 1 α

None of the Phase 5 code has run against the real model. Prove the path end-to-end on a handful
of prompts before spending an hour of GPU on ~1700 generations: never debug generation code with
the pod live.

In [ ]:
import json, yaml
L      = R("layer_select")["best_layer"]
alphas = R("alpha_ladder")["alphas"]

rows = [json.loads(l) for l in open("data/eval/fiction_jailbreaks.jsonl")][:3]
with open("data/eval/_smoke.jsonl", "w") as f:
    for r in rows: f.write(json.dumps(r) + "\n")

cfg_raw = yaml.safe_load(open("configs/qwen.yaml"))     # same knobs, tiny prompt set
cfg_raw["data"]["smoke"] = "data/eval/_smoke.jsonl"
cfg_raw["generate"]["max_new_tokens"] = 64
yaml.safe_dump(cfg_raw, open("configs/_smoke.yaml", "w"), sort_keys=False)

step("05_generate.py", "--config", "configs/_smoke.yaml", "--eval", "smoke",
     "--alphas", alphas[1], "--force")
print("--- sample generations ---")
for r in [json.loads(l) for l in open(f"artifacts/generations/smoke_L{L}.jsonl")][:4]:
    print(f"[{r['condition']} alpha={r.get('alpha')}]", r["completion"][:160].replace("\n"," "))

## 17 · The full sweep — GPU, the long one

Layer and ladder come from the stored results (`auto`). Resumable per (condition, α), so a
disconnect costs only the conditions that had not finished, and the ladder actually used is
recorded in the `.meta.json` sidecar.

Rough size: ~100 attacks × (1 baseline + 4 conditions × 4 α) ≈ 1700 generations, plus XSTest.
Run the attack sweep first; if time is short, XSTest at the single headline α is the part you
cannot skip — without it "restores refusal" is unsupported. **Then stop the GPU**: judging and
analysis are API/CPU work.

In [ ]:
step("05_generate.py", "--eval", "fiction_jailbreaks")
step("05_generate.py", "--eval", "xstest")
!wc -l artifacts/generations/*.jsonl

## 18 · Read 20 raw generations BEFORE any aggregate
Every real bug in this pipeline is visible in 20 samples and invisible in a bar chart. Is the
"refusal" actually a refusal, or a truncated sentence? Is the "bypass" actually harmful, or the
model misunderstanding the prompt?

In [ ]:
import json, random
L = R("layer_select")["best_layer"]
rows = [json.loads(l) for l in open(f"artifacts/generations/fiction_jailbreaks_L{L}.jsonl")]
random.Random(0).shuffle(rows)
for r in rows[:20]:
    print(f"--- {r['condition']} alpha={r.get('alpha')} ---")
    print(r["completion"][:300].replace("\n", " "), "\n")

## 19 · Judge and analyse — API, no GPU
Three-way labels via the Batch API. Needs `ANTHROPIC_API_KEY`. The analysis applies the four
readings — null-explains-it, one-sided effect, degeneration-not-defense, nothing-moved — and
refuses to bless a result without the XSTest over-refusal guard.

In [ ]:
import os, getpass
if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("ANTHROPIC_API_KEY: ")

L = R("layer_select")["best_layer"]
step("06_judge.py", "--generations", f"artifacts/generations/fiction_jailbreaks_L{L}.jsonl")
step("06_judge.py", "--generations", f"artifacts/generations/xstest_L{L}.jsonl")
step("08_sweep_analysis.py",
     "--scores",  f"artifacts/scores/fiction_jailbreaks_L{L}_scored.jsonl",
     "--xstest",  f"artifacts/scores/xstest_L{L}_scored.jsonl")

**Spot-check the judge before believing it.** An unvalidated LLM judge is an unmeasured
instrument. Read ~50 judgments against the rubric by hand and report the agreement rate in the
write-up — if the judge and you disagree often, the three-way numbers mean little.

In [ ]:
import json, random
L = R("layer_select")["best_layer"]
scored = [json.loads(l) for l in open(f"artifacts/scores/fiction_jailbreaks_L{L}_scored.jsonl")]
random.Random(1).shuffle(scored)
for r in scored[:10]:
    print(f"[{r.get('label')}] SR={r.get('strongreject')} coherent={r.get('coherent')}")
    print("   ", (r.get("completion") or "")[:200].replace("\n", " "), "\n")
print("Agreement rate you measured by hand: ____ / 50   <- put this in the write-up")

## 20 · Save everything, and what this shows

* A high held-out AUC means the information is **present and linearly readable** on unseen
  framings. It is **not** causal evidence — that is the Phase 5 steering experiment.
* Distinctness from `r_hat` is only half of Claim 1; the sharper test is `v_C` vs `v_MP`,
  because "the model is play-acting" is the most plausible deflationary explanation.
* Beating a text baseline is a *number*, not a category. State how close the best surface
  baseline got; do not write "beyond surface vocabulary" if it reached 85% of the way.
* Scope every claim to this real-vs-hypothetical contrast set, this model, and the specific
  attacks tested — jailbreak-relevant features are attack-family-specific (Kirch et al.).

Every number above is in `artifacts/results/*.json` with the git SHA, the input hashes and the
command that produced it. That is what the write-up cites, and what the facilitators can check.

**Update `logs/research_log.md` before closing the session.**

In [ ]:
if MOUNT:
    print("backed up", sync("artifacts", DEST), "files ->", DEST)
!ls artifacts/results/